# 17 — Does a 100–200-trial search improve longer-history LightGBM?

Use the verified project kernel and **Restart Kernel and Run All** after the original 24-trial 09b study. Notebooks 15/16 are not prerequisites. No preparation or downloads run here. Keep the existing local data/cache and parent study.

**Controlled question:** does a larger Optuna budget help the **original 22-feature longer-history model**, with the same six-parameter search space, three expanding folds, 10% target and evaluation rows? This does not add the features from 15/16.

Default: **200 total trials, including the 24 completed parent trials**. Report the best candidate after 24, 100 and 200. At most 176 new configurations × three folds = 528 new fold fits, with duplicates reused. The new continuation uses a seeded TPE sampler initialised with the recorded trials; the original sampler's internal random state was not saved, so this is not a bit-for-bit continuation of its random sequence.

Checkpoint each completed trial. Restart/Run All resumes the same compatible search. You may initially set `TARGET_TRIALS = 100`, then extend to 200 without repeating completed trials. Default 200 evaluates later validation only after all 200 finish. Do not change the feature set/search ranges/seed to chase a preferred validation result.

Mean fold AP selects each budget's candidate; only then read January–March validation and report all budget results. The final research candidate is the largest completed budget's fold winner, **not whichever row has highest validation AP**. These periods were already inspected and remain exploratory. No April–June test or registered November outcome is read. No dashboard replacement occurs. More trials do not guarantee 0.70 AP or independent future improvement.

## 1. Verify the 24-trial parent and declare the budget

The original saved models and data remain read-only. Use their exact Python/library versions. An incompatible or missing input stops the study rather than rebuilding preparation. Budget is the only user setting below.

In [ ]:
import sys
from pathlib import Path
REPOSITORY_ROOT = Path.cwd().resolve()
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open this notebook inside the FYP repository.")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import ast
import optuna
from tempfile import TemporaryDirectory
import hashlib
import json
from datetime import datetime, timezone
from importlib import metadata, reload
from time import perf_counter
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_recall_curve,
                             precision_score, recall_score, f1_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier
import pricecatcher_config as cfg
cfg = reload(cfg)

def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

history_runs = sorted((cfg.PROJECT_ROOT / "09b_longer_history_lightgbm").glob("*/tuned/study_manifest.json"), reverse=True)
if not history_runs:
    raise FileNotFoundError("Complete 09b_longer_history_lightgbm.ipynb first.")
control_dir = history_runs[0].parent.parent / "optuna"
if not (control_dir / "study_manifest.json").is_file():
    raise FileNotFoundError("The latest 09b run needs its completed direct-history Optuna result.")
parent = cfg.verify_saved_study(control_dir)
selection = parent["selection"]
if selection["family"] != "LightGBM" or selection["training_start"] != "2024-01":
    raise ValueError("Expected the completed longer-history LightGBM control.")
if sys.version_info[:2] != tuple(map(int, parent["python_version"].split(".")[:2])):
    raise ValueError("Use the same Python major/minor version as 09b.")
for package in ["optuna", "lightgbm", "scikit-learn", "numpy", "pandas", "joblib"]:
    if metadata.version(package) != parent["package_versions"][package]:
        raise ValueError(f"Use the saved 09b {package} version for this controlled comparison.")

FOLDS = ["fold_1", "fold_2", "fold_3"]
folders = {name: cfg.PROJECT_ROOT / "experiments" / name / "05_modelling" for name in FOLDS + ["holdout"]}
settings_by_fold = {name: json.loads((folder / "experiment_settings.json").read_text()) for name, folder in folders.items()}
settings = settings_by_fold["holdout"]
if any(settings["preparation"][key] != value for key, value in cfg.PREPARATION_SETTINGS.items()):
    raise ValueError("Current configuration differs from the saved preparation. Restore its matching configuration.")
FEATURES, NUMERIC, CATEGORICAL = (settings[key] for key in ["feature_columns", "numeric_features", "categorical_features"])
if FEATURES != selection["feature_columns"] or settings["surge_threshold_pct"] != 10:
    raise ValueError("Keep the control feature list and 10% definition unchanged.")
MODEL_PARAMETERS = {**selection["fixed_parameters"], **selection["parameters"],
                    "random_state": selection["seed"], "n_jobs": selection["n_jobs"]}
control_model = joblib.load(control_dir / "lightgbm.joblib")
fitted_parameters = control_model.named_steps["model"].get_params()
if any(fitted_parameters[key] != value for key, value in MODEL_PARAMETERS.items()):
    raise ValueError("Declared model settings differ from the saved control model.")
parent_manifest_sha256 = file_hash(control_dir / "study_manifest.json")
configuration_sha256 = file_hash(Path(cfg.__file__))
print("Active run:", cfg.PROJECT_ROOT)
print("Frozen control:", control_dir)
print("Parameters (unchanged):", MODEL_PARAMETERS)

TARGET_TRIALS = 200  # Choose 100 or 200 TOTAL trials, including the parent's 24.
if TARGET_TRIALS not in [100, 200]:
    raise ValueError("Use the declared 100- or 200-trial budget.")
if selection["trials"] != 24:
    raise ValueError("This experiment expects the original completed 24-trial 09b parent.")
SEED = selection["seed"]
SEARCH_SPACE = selection["search"]  # Exactly the original ranges; do not widen them here.
print("Total trial budget:", TARGET_TRIALS, "| inherited:", selection["trials"])
print("Search space (unchanged):", SEARCH_SPACE)

## 2. Reuse the same longer-history preparation and model pipeline

Only the two reviewed training functions are loaded from 09b. They reproduce the original 2025 rows before adding compatible 2024 history. No earlier notebook cells execute. Rebuilt training support must match the parent. Preprocessing is fitted afresh in every new fold fit.

In [ ]:
history_notebook = REPOSITORY_ROOT / "09b_longer_history_lightgbm.ipynb"
source_nb = json.loads(history_notebook.read_text())
functions = []
for cell in source_nb["cells"]:
    if cell["cell_type"] == "code":
        tree = ast.parse("".join(cell["source"]))
        functions.extend(node for node in tree.body if isinstance(node, ast.FunctionDef)
                         and node.name in {"training_table", "prepare_training"})
if sorted(node.name for node in functions) != ["prepare_training", "training_table"]:
    raise ValueError("09b training helpers changed; review compatibility before this study.")
calculation = ast.Module(body=functions, type_ignores=[])
helper_sha256 = hashlib.sha256(ast.dump(calculation).encode()).hexdigest()
exec(compile(calculation, str(history_notebook), "exec"), globals())
helper_notebook_sha256 = file_hash(history_notebook)

additional_inputs = parent["additional_input_sha256"]
historical_paths = [cfg.resolve_input(path) for path in additional_inputs
                    if cfg.resolve_input(path).name == "selected_prices_2024.parquet"]
if len(historical_paths) != 1:
    raise ValueError("Expected the exact 2024 source snapshot recorded by 09b.")
earlier_source = pd.read_parquet(historical_paths[0], columns=cfg.GROUP_KEYS + ["date", "premise_code", "price"])
earlier_source["date"] = pd.to_datetime(earlier_source.date)
if earlier_source.empty or not earlier_source.date.between("2024-01-01", "2024-12-31").all():
    raise ValueError("Supplementary source must contain only the recorded 2024 observations.")
features, keys = FEATURES, cfg.GROUP_KEYS + ["week_start"]
training_rows, cleaning_rows = [], []
expected_support = pd.read_csv(control_dir / "training_comparison.csv")

def matching_training(name):
    training, evaluation = prepare_training(name)
    for history, recorded in [("2025-only", training_rows[-2]), ("2024-added", training_rows[-1])]:
        expected = expected_support.loc[expected_support.fold.eq(name) & expected_support.history.eq(history)]
        if len(expected) != 1:
            raise ValueError(f"{name}: missing original support evidence.")
        for column in ["rows", "positives", "first_week", "last_outcome"]:
            if str(recorded[column]) != str(expected.iloc[0][column]):
                raise ValueError(f"{name}: rebuilt {history} {column} differs from 09b.")
    if training.surge_label.nunique() != 2 or evaluation.surge_label.nunique() != 2:
        raise ValueError("Both classes are needed for the overall AP comparison.")
    return training, evaluation

In [ ]:
def model_inputs(frame):
    X = frame[FEATURES].copy()
    for column in CATEGORICAL:
        X[column] = X[column].astype("string").fillna("__missing__").astype(str)
    return X

def fit_candidate(parameters, training, evaluation):
    preprocess = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), NUMERIC),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL),
    ], sparse_threshold=1.0)
    fixed = {**selection["fixed_parameters"], "random_state": SEED, "n_jobs": selection["n_jobs"]}
    model = Pipeline([("preprocess", preprocess), ("model", LGBMClassifier(**{**fixed, **parameters}))])
    model.fit(model_inputs(training), training.surge_label.astype(int))
    scores = model.predict_proba(model_inputs(evaluation))[:, 1]
    if not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all():
        raise ValueError("Invalid model scores.")
    return model, scores

def evaluate(y, scores, cutoff):
    y = np.asarray(y, dtype=int)
    alerts = np.asarray(scores) >= cutoff
    tn, fp, fn, tp = confusion_matrix(y, alerts, labels=[0, 1]).ravel()
    return {"rows": len(y), "positives": int(y.sum()), "average_precision":
        float(average_precision_score(y, scores)) if len(np.unique(y)) == 2 else np.nan,
        "cutoff": cutoff, "precision": precision_score(y, alerts, zero_division=0),
        "recall": recall_score(y, alerts, zero_division=0) if y.sum() else np.nan,
        "f1": f1_score(y, alerts, zero_division=0) if y.sum() else np.nan,
        "alerts": int(alerts.sum()), "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}

## 3. Restore the original trials and compatible progress

The parent CSV records all 24 completed configurations and their three fold metrics. Check the AP summaries, parameters and population counts before reusing them. The new sampler learns from these observations; they are not fitted again.

A deterministic study folder identifies the parent, source calculations, configuration and software. `checkpoint.joblib` stores the study/sampler and completed fold results after each successful trial. Load only this locally generated trusted checkpoint. Source/code changes start a separate search identity. Changing only 100 → 200 continues the existing one. Previous completed result folders remain intact.

In [ ]:
this_notebook = REPOSITORY_ROOT / "research" / "17_larger_optuna_budget.ipynb"

def calculation_hash():
    notebook = json.loads(this_notebook.read_text())
    tree = ast.parse("\n".join("".join(c["source"]) for c in notebook["cells"] if c["cell_type"] == "code"))
    # Budget changes are allowed when extending 100 to 200; other code stays fixed.
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "TARGET_TRIALS" for t in node.targets):
            node.value = ast.Constant(value=0)
    return hashlib.sha256(ast.dump(tree).encode()).hexdigest()

packages = {p: metadata.version(p) for p in ["optuna", "lightgbm", "scikit-learn", "numpy", "pandas", "joblib"]}
identity = {"parent_sha256": parent_manifest_sha256, "helper_sha256": helper_sha256,
    "configuration_sha256": configuration_sha256, "code_sha256": calculation_hash(),
    "python_version": sys.version.split()[0], "package_versions": packages,
    "search_space": SEARCH_SPACE, "seed": SEED, "features": FEATURES,
    "sampler_policy": "new seeded TPE conditioned on 24 recorded trials; saved sampler state on resume"}
study_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
work = cfg.PROJECT_ROOT / "research" / "17_larger_optuna_budget" / study_id
work.mkdir(parents=True, exist_ok=True)
checkpoint = work / "checkpoint.joblib"
distributions = {
    "learning_rate": optuna.distributions.FloatDistribution(**SEARCH_SPACE["learning_rate"]),
    "n_estimators": optuna.distributions.IntDistribution(**SEARCH_SPACE["n_estimators"]),
    "num_leaves": optuna.distributions.CategoricalDistribution(SEARCH_SPACE["num_leaves"]),
    "min_child_samples": optuna.distributions.CategoricalDistribution(SEARCH_SPACE["min_child_samples"]),
    "reg_lambda": optuna.distributions.FloatDistribution(**SEARCH_SPACE["reg_lambda"]),
    "class_weight": optuna.distributions.CategoricalDistribution(SEARCH_SPACE["class_weight"]),
}

def parameters_from_row(row):
    return {"learning_rate": float(row.learning_rate), "n_estimators": int(row.n_estimators),
        "num_leaves": int(row.num_leaves), "min_child_samples": int(row.min_child_samples),
        "reg_lambda": float(row.reg_lambda), "class_weight": None if row.class_weight == "none" else row.class_weight}

original_summary = pd.read_csv(control_dir / "trial_summary.csv", keep_default_na=False, float_precision="round_trip")
original_folds = pd.read_csv(control_dir / "trial_fold_metrics.csv", float_precision="round_trip", keep_default_na=False)
if sorted(original_summary.trial.tolist()) != list(range(24)):
    raise ValueError("Expected 24 consecutive parent trials.")
if checkpoint.exists():
    saved = joblib.load(checkpoint)
    if saved["identity"] != identity:
        raise ValueError("Incompatible checkpoint. Preserve it and investigate the changed inputs/code.")
    study, trial_rows = saved["study"], saved["trial_rows"]
else:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.NopPruner())
    trial_rows = original_folds.to_dict(orient="records")
    for row in original_summary.sort_values("trial").itertuples():
        study.add_trial(optuna.trial.create_trial(params=parameters_from_row(row), distributions=distributions,
            value=float(row.mean_ap), user_attrs={"inherited_from_09b": True}))

# Reconstruct cache from recorded metrics; duplicate proposals can reuse scores.
trial_cache = {}
recorded = pd.DataFrame(trial_rows)
if len(recorded) != 3 * len(study.trials) or len(study.trials) < 24:
    raise ValueError("Checkpoint trials and fold rows disagree.")
for trial in study.trials:
    if trial.state != optuna.trial.TrialState.COMPLETE:
        raise ValueError("Checkpoint must contain only completed trials.")
    rows = recorded.loc[recorded.trial.eq(trial.number)]
    if len(rows) != 3 or set(rows.fold) != set(FOLDS):
        raise ValueError("Each trial needs exactly three fold results.")
    np.testing.assert_allclose(rows.average_precision.mean(), trial.value, rtol=1e-10, atol=1e-12)
    for row in rows.itertuples():
        if parameters_from_row(row) != trial.params:
            raise ValueError("Saved trial parameters and fold results disagree.")
    trial_cache.setdefault(json.dumps(trial.params, sort_keys=True), rows.to_dict(orient="records"))
for row in original_summary.itertuples():
    trial = study.trials[row.trial]
    if trial.params != parameters_from_row(row):
        raise ValueError("Checkpoint changed an inherited trial configuration.")
    np.testing.assert_allclose(trial.value, row.mean_ap, rtol=1e-12, atol=1e-12)
    rows = original_folds.loc[original_folds.trial.eq(row.trial)]
    np.testing.assert_allclose([rows.average_precision.mean(), rows.average_precision.std()],
                              [row.mean_ap, row.std_ap], rtol=1e-10, atol=1e-12)
parent_best = max(study.trials[:24], key=lambda t: (t.value, -t.number))
if parent_best.number != selection["selected_trial"] or parent_best.params != selection["parameters"]:
    raise ValueError("Inherited winner differs from the recorded 09b selection.")
if len(study.trials) > TARGET_TRIALS:
    raise ValueError("This checkpoint already exceeds the requested budget. Use its existing larger budget.")
print("Study folder:", work)
print("Completed:", len(study.trials), "| remaining:", TARGET_TRIALS - len(study.trials))

## 4. Continue the chronological search and checkpoint each trial

Temporary training files avoid retaining three large expanded training frames in memory. Rebuild only the three folds if additional trials are needed. January–March validation is not loaded here. No early stopping/pruning or 0.70-based stopping is used. Any failure leaves the last successful checkpoint available; fix the error and Restart/Run All to resume.

Only successful trials are committed. If interruption occurs during a trial, that incomplete trial can be repeated after restart. A checkpointed completed trial is reused. The final AP progress curve is a selected search score, not an unbiased accuracy estimate.

In [ ]:
def save_checkpoint():
    temporary = checkpoint.with_suffix(".tmp")
    joblib.dump({"identity": identity, "study": study, "trial_rows": trial_rows}, temporary)
    temporary.replace(checkpoint)

def objective(trial):
    parameters = {
        "learning_rate": trial.suggest_float("learning_rate", **SEARCH_SPACE["learning_rate"]),
        "n_estimators": trial.suggest_int("n_estimators", **SEARCH_SPACE["n_estimators"]),
        "num_leaves": trial.suggest_categorical("num_leaves", SEARCH_SPACE["num_leaves"]),
        "min_child_samples": trial.suggest_categorical("min_child_samples", SEARCH_SPACE["min_child_samples"]),
        "reg_lambda": trial.suggest_float("reg_lambda", **SEARCH_SPACE["reg_lambda"]),
        "class_weight": trial.suggest_categorical("class_weight", SEARCH_SPACE["class_weight"]),
    }
    key = json.dumps(parameters, sort_keys=True)
    if key in trial_cache:
        rows = [{**row, "trial": trial.number, "reused_from_trial": row["trial"],
                 "fit_and_score_seconds": 0.0} for row in trial_cache[key]]
    else:
        rows = []
        for name in FOLDS:
            training = pd.read_parquet(training_paths[name])
            evaluation = fold_validation[name]
            started = perf_counter()
            model, scores = fit_candidate(parameters, training, evaluation)
            rows.append({"trial": trial.number, "fold": name, **parameters,
                "class_weight": parameters["class_weight"] or "none",
                "fit_and_score_seconds": perf_counter() - started,
                **evaluate(evaluation.surge_label, scores, 0.5)})
            del training, model
        trial_cache[key] = rows
    trial_rows.extend(rows)
    return float(np.mean([row["average_precision"] for row in rows]))

save_checkpoint()
if len(study.trials) < TARGET_TRIALS:
    with TemporaryDirectory(prefix="hargalens_budget_") as temporary:
        training_paths, fold_validation = {}, {}
        for name in FOLDS:
            training, evaluation = matching_training(name)
            training_paths[name] = Path(temporary) / (name + ".parquet")
            training[FEATURES + ["surge_label"]].to_parquet(training_paths[name], index=False)
            fold_validation[name] = evaluation
            # Every inherited/reused fold must describe this exact population.
            old = original_folds.loc[original_folds.fold.eq(name)]
            if not old.rows.eq(len(evaluation)).all() or not old.positives.eq(int(evaluation.surge_label.sum())).all():
                raise ValueError("Parent and rebuilt fold populations differ.")
            del training
        while len(study.trials) < TARGET_TRIALS:
            trial = study.ask()
            value = objective(trial)
            study.tell(trial, value)
            save_checkpoint()
            print(f"Completed {len(study.trials)}/{TARGET_TRIALS} | trial AP {value:.4f} | best {study.best_value:.4f}", flush=True)
        del fold_validation

budgets = [n for n in [24, 100, 200] if n <= TARGET_TRIALS]
winners = {budget: max(study.trials[:budget], key=lambda t: (t.value, -t.number)) for budget in budgets}
trial_folds = pd.DataFrame(trial_rows)
fold_summary = pd.DataFrame([{"total_trials": budget, "selected_trial": trial.number,
    "mean_fold_ap": trial.value,
    "std_fold_ap": trial_folds.loc[trial_folds.trial.eq(trial.number), "average_precision"].std(),
    "parameters": json.dumps(trial.params, sort_keys=True)} for budget, trial in winners.items()])
display(fold_summary)
print("Budgets and winners frozen before Jan–Mar validation. Highest-budget winner remains the research candidate.")

## 5. Compare all budget winners on later validation

Only now rebuild training through December 2025 and read January–March 2026. The 24-trial control must reproduce its saved scores and exact identifiers/labels. Fit the 100/200 winners only when their settings differ from an already fitted candidate. Evaluate all budgets even if the later scores decline. Each chooses its own F1 cutoff on this same validation period; these are apparent selection-period metrics. No validation score changes the fold-selected winner.

In [ ]:
training, validation = matching_training("holdout")
prediction_keys = ["item_code", "state", "district", "premise_type", "project_group",
                   "week_start", "next_week_end", "surge_label"]
identifiers = prediction_keys[:-1]
validation = validation.sort_values(identifiers).reset_index(drop=True)
old_predictions = pd.read_parquet(control_dir / "validation_predictions.parquet").sort_values(identifiers).reset_index(drop=True)
if validation.duplicated(identifiers).any() or old_predictions.duplicated(identifiers).any():
    raise ValueError("Duplicate validation identifiers.")
pd.testing.assert_frame_equal(validation[prediction_keys], old_predictions[prediction_keys])
control_scores = control_model.predict_proba(model_inputs(validation))[:, 1]
np.testing.assert_allclose(control_scores, old_predictions.model_score, rtol=1e-9, atol=1e-11)
fitted = {json.dumps(selection["parameters"], sort_keys=True): (control_model, control_scores, float(selection["score_cutoff"]))}
validation_rows, group_rows, comparison_rows = [], [], []
predictions = validation[prediction_keys].copy()
for budget, trial in winners.items():
    key = json.dumps(trial.params, sort_keys=True)
    if key not in fitted:
        model, scores = fit_candidate(trial.params, training, validation)
        p, r, cutoffs = precision_recall_curve(validation.surge_label, scores)
        f1 = np.divide(2*p[:-1]*r[:-1], p[:-1]+r[:-1], out=np.zeros(len(cutoffs)), where=(p[:-1]+r[:-1]) > 0)
        cutoff = float(cutoffs[np.flatnonzero(np.isclose(f1, f1.max(), rtol=0, atol=1e-12))[-1]])
        fitted[key] = (model, scores, cutoff)
    model, scores, cutoff = fitted[key]
    for role, threshold in [("common 0.5 reference", 0.5), ("apparent cutoff-selection performance", cutoff)]:
        validation_rows.append({"total_trials": budget, "selected_trial": trial.number, "role": role,
                                **evaluate(validation.surge_label, scores, threshold)})
    comparison_rows.append({"total_trials": budget, "selected_trial": trial.number, "mean_fold_ap": trial.value,
                            **evaluate(validation.surge_label, scores, cutoff)})
    for group, positions in validation.groupby("project_group", sort=True).indices.items():
        group_rows.append({"total_trials": budget, "project_group": group,
            **evaluate(validation.iloc[positions].surge_label, scores[positions], cutoff)})
    predictions[f"score_{budget}"] = scores
    predictions[f"alert_{budget}"] = scores >= cutoff
final_trial = winners[TARGET_TRIALS]
final_model, _, final_cutoff = fitted[json.dumps(final_trial.params, sort_keys=True)]
budget_comparison = pd.DataFrame(comparison_rows)
validation_comparison = pd.DataFrame(validation_rows)
validation_groups = pd.DataFrame(group_rows)
display(budget_comparison)
display(validation_groups)
del training
print("Final research candidate:", TARGET_TRIALS, "trials | selected trial:", final_trial.number)
print("No dashboard change or independent final-test claim follows from these results.")

## 6. Save a separate immutable result and search-progress plot

The resumable checkpoint remains in the study folder. Each completed evaluation writes a new `results/<timestamp>/`, containing a snapshot of its trials, comparisons, predictions and highest-budget model. Extending the checkpoint later does not alter a previously completed result. The manifest records hashes and selection rules; the dashboard pin remains untouched.

In [ ]:
cfg.verify_saved_study(control_dir)
if (file_hash(control_dir / "study_manifest.json") != parent_manifest_sha256
        or file_hash(history_notebook) != helper_notebook_sha256
        or file_hash(Path(cfg.__file__)) != configuration_sha256 or calculation_hash() != identity["code_sha256"]):
    raise ValueError("Upstream study/calculation changed during this search.")
output = work / "results" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
output.mkdir(parents=True, exist_ok=False)
trial_summary = pd.DataFrame([{"trial": t.number, "mean_fold_ap": t.value, **t.params,
    "class_weight": t.params["class_weight"] or "none", "inherited": t.number < 24} for t in study.trials])
trial_summary["best_fold_ap_so_far"] = trial_summary.mean_fold_ap.cummax()
for name, table in {"budget_comparison.csv": budget_comparison, "fold_summary.csv": fold_summary,
    "trial_summary.csv": trial_summary, "trial_fold_metrics.csv": trial_folds,
    "validation_comparison.csv": validation_comparison, "validation_group_metrics.csv": validation_groups,
    "training_support.csv": expected_support}.items():
    table.to_csv(output / name, index=False)
predictions.to_parquet(output / "validation_predictions.parquet", index=False)
joblib.dump(final_model, output / "research_model.joblib")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(trial_summary.trial + 1, trial_summary.best_fold_ap_so_far)
axes[0].axhline(0.70, linestyle=":", color="grey", label="0.70 aspiration")
axes[0].set(xlabel="Completed trials (including inherited 24)", ylabel="Selected mean fold AP", ylim=(0, 1), title="Search score, not independent accuracy")
axes[0].legend()
axes[1].plot(budget_comparison.total_trials, budget_comparison.mean_fold_ap, "o-", label="Selected fold AP")
axes[1].plot(budget_comparison.total_trials, budget_comparison.average_precision, "o-", label="Jan–Mar validation AP")
axes[1].set(xlabel="Total search budget", ylabel="Average precision", ylim=(0, 1), title="More trials need not generalise better")
axes[1].legend(); fig.tight_layout(); fig.savefig(output / "budget_comparison.png", dpi=150); plt.show()
manifest = {"status": "completed", "identity": identity, "target_trials": TARGET_TRIALS,
    "inherited_trials": 24, "new_recorded_trials": TARGET_TRIALS - 24, "selected_trial": final_trial.number,
    "parameters": final_trial.params, "fixed_parameters": selection["fixed_parameters"],
    "score_cutoff": final_cutoff, "surge_threshold_pct": 10.0, "feature_columns": FEATURES,
    "selection_rule": "highest mean fold AP in highest budget; earliest trial breaks ties; no validation reselection",
    "validation_role": "already-inspected exploratory comparison and F1 cutoff selection",
    "dashboard_changed": False, "independent_test_evaluated": False,
    "parent_study": cfg.input_reference(control_dir / "study_manifest.json"),
    "parent_manifest_sha256": parent_manifest_sha256, "input_sha256": parent["input_sha256"],
    "additional_input_sha256": parent["additional_input_sha256"],
    "output_sha256": {p.name: file_hash(p) for p in output.iterdir()}}
(output / "research_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Saved:", output)
print("Share budget_comparison.csv, fold_summary.csv, trial_summary.csv and validation_group_metrics.csv.")

## 7. Interpret the larger budget honestly

- Best fold AP cannot decrease when earlier trials are retained. This is a selection property, not proof of better future predictions.
- Compare later AP, alert counts and group support at 24/100/200 even if they worsen. The highest-budget fold winner stays the research candidate; do not pick a smaller budget retrospectively because its validation score is larger.
- A flat progress curve suggests diminishing returns within this fixed space. More hours, trees or trials are not themselves evidence of quality.
- A gain on inspected periods still needs independent confirmation. Do not call a small increase statistically significant without an appropriate analysis of temporal/dependent observations.
- If interrupted, restart this notebook with the same budget/kernel. To extend a completed 100-trial run, change only the budget to 200. Existing 100-trial outputs remain archived.

No change to the deployed model is made. A deployment decision would need its own checkpoint review and matching explanation/export integration. AP above 0.70 remains unguaranteed.